# Comparacion de graficas: Python original vs C

Esta libreta carga los CSV generados por `pitch_optimo.c` y reconstruye las mismas variables usadas por `Pitch_optimo.ipynb` para graficar con el mismo formato.

La idea es que estas graficas se puedan comparar directamente contra las graficas de la libreta original.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

%matplotlib inline

## Cargar CSV generados por C

In [ ]:
empuje_csv = pd.read_csv("empuje_vs_vd.csv")
rpm_csv = pd.read_csv("rpm_vs_vd.csv")
eficiencia_csv = pd.read_csv("eficiencia.csv")
vd_eq_csv = pd.read_csv("vd_equilibrio.csv")
superficie_csv = pd.read_csv("superficie_3d.csv")

empuje_csv.head()

## Reconstruir variables con los mismos nombres de `Pitch_optimo.ipynb`

In [ ]:
# Variables equivalentes a la libreta original.
Vd_sensor = empuje_csv["Vd_m_s"].to_numpy()
alpha_cols = empuje_csv.columns[1:]
alpha_deg = np.array([float(col.replace("alpha_", "")) for col in alpha_cols])

n_alpha = len(alpha_deg)
n_Vd = len(Vd_sensor)

# En la libreta original: T_matrix tiene shape (n_alpha, n_Vd).
# En el CSV: cada columna es un alpha y cada renglon es una Vd.
T_matrix = empuje_csv[alpha_cols].to_numpy().T

# El CSV rpm_vs_vd ya trae RPM. Para conservar el mismo codigo de graficado
# original, reconstruimos Om en rad/s y luego lo volvemos a convertir a RPM.
RPM_matrix = rpm_csv[alpha_cols].to_numpy().T
Om = RPM_matrix * (2 * np.pi) / 60

eficiencia = eficiencia_csv["CL_CD"].to_numpy()
Vd_eq = pd.to_numeric(vd_eq_csv["Vd_eq_m_s"], errors="coerce").to_numpy()

print(f"n_alpha = {n_alpha}")
print(f"n_Vd = {n_Vd}")
print(f"T_matrix shape = {T_matrix.shape}")
print(f"Om shape = {Om.shape}")

## 1. Empuje vs velocidad de descenso

In [ ]:
colores = plt.cm.tab20(np.linspace(0, 1, n_alpha))
 
# 1. Empuje vs Velocidad de descenso
fig1, ax1 = plt.subplots(figsize=(10, 6))
for i in range(n_alpha):
    ax1.plot(Vd_sensor, T_matrix[i, :], linewidth=2, color=colores[i],
             label=f'α = {alpha_deg[i]:.2f}°')
ax1.set_xlabel('Velocidad de descenso [m/s]', fontsize=10)
ax1.set_ylabel('Empuje T [N]', fontsize=10)
ax1.set_title('Empuje vs Velocidad de Descenso', fontweight='bold')
ax1.legend(loc='upper left', fontsize=7, ncol=2)
ax1.set_xlim([0, 20])
ax1.set_ylim([0, T_matrix.max()])
ax1.grid(True)
plt.tight_layout()

## 2. RPM vs velocidad de descenso

In [ ]:
# 2. RPM vs Velocidad de descenso
fig2, ax2 = plt.subplots(figsize=(10, 6))
for i in range(n_alpha):
    RPM_line = Om[i, :] * 60 / (2 * np.pi)
    ax2.plot(Vd_sensor, RPM_line, linewidth=2, color=colores[i],
             label=f'α = {alpha_deg[i]:.2f}°')
ax2.set_xlabel('Velocidad de descenso [m/s]', fontsize=10)
ax2.set_ylabel('Velocidad angular Ω [RPM]', fontsize=10)
ax2.set_title('RPM vs Velocidad de descenso', fontweight='bold')
ax2.legend(loc='upper left', fontsize=7, ncol=2)
ax2.set_xlim([0, 20])
ax2.grid(True)
plt.tight_layout()

## 3. Eficiencia aerodinamica por angulo

In [ ]:
# 3. Eficiencia aerodinámica por ángulo
fig3, ax3 = plt.subplots(figsize=(8, 5))
bars = ax3.bar(alpha_deg, eficiencia, color=[0.18, 0.37, 0.54], edgecolor='none')
idx_max = np.argmax(eficiencia)
bars[idx_max].set_facecolor([0.91, 0.49, 0.13])
ax3.set_xlabel('Ángulo de pitch α [°]', fontsize=10)
ax3.set_ylabel('Eficiencia CL / CD', fontsize=10)
ax3.set_title('Eficiencia Aerodinámica por Ángulo', fontweight='bold')
ax3.grid(True)
plt.tight_layout()

## 4. Velocidad de descenso de equilibrio vs pitch

In [ ]:
# 4. Velocidad de descenso de equilibrio vs Pitch
fig4, ax4=plt.subplots(figsize=(8, 5))
ax4.plot(alpha_deg, Vd_eq, 'o-', linewidth=2,color=[0.18, 0.37, 0.54], markersize=8,markerfacecolor=[0.91, 0.49, 0.13])
ax4.set_xlabel('Ángulo de pitch α [°]', fontsize=10)
ax4.set_ylabel('Vd equilibrio [m/s]', fontsize=10)
ax4.set_title('Velocidad de Descenso de Equilibrio vs Pitch', fontweight='bold')
ax4.grid(True, which='both')
ax4.minorticks_on()
plt.tight_layout()

## 5. Superficie 3D: RPM, pitch y velocidad de descenso

In [ ]:
# 5. Superficie 3D: RPM, Pitch, Velocidad de descenso
Alpha_mesh, Vd_mesh = np.meshgrid(alpha_deg, Vd_sensor)   # shapes: (n_Vd, n_alpha)
Om_mesh = Om.T                                            # transpose → (n_Vd, n_alpha)
 
fig5= plt.figure(figsize=(10,7))
ax5=fig5.add_subplot(111, projection='3d')
surf=ax5.plot_surface(Alpha_mesh, Vd_mesh, Om_mesh, cmap='turbo', linewidth=0, antialiased=True)
ax5.set_xlabel('Pitch (α) [°]')
ax5.set_ylabel('Vel. descenso [m/s]')
ax5.set_zlabel('Vel. angular Ω [rad/s]')
ax5.set_title('Relación entre Pitch, RPM y Vel. de descenso', fontweight='bold')
cb = fig5.colorbar(surf, ax=ax5, shrink=0.5, pad=0.1)
cb.set_label('Velocidad angular Ω [rad/s]')
plt.tight_layout()
 
plt.show()